## 1. Environment Setup & Dependency Installation
ပထမဦးဆုံးအဆင့်အနေနဲ့ KITTI Data တွေကို ကိုင်တွယ်ဖို့နဲ့ Vector သင်္ချာတွက်ချက်မှုတွေ ပြုလုပ်ဖို့ လိုအပ်တဲ့ Computer Vision Libraries များနှင့် Kagglehub Engine ကို Install လုပ်ပြီး Import လုပ်ပါမယ်။

In [1]:
!nvidia-smi

Wed Jul  1 08:17:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.58.03              Driver Version: 595.58.03      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:42:00.0 Off |                  Off |
|  0%   52C    P8             11W /  400W |       1MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Install required libraries if not available
%pip install -q numpy pandas opencv-python matplotlib tqdm kagglehub

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import kagglehub

print("✅ Libraries and Kagglehub Imported Successfully!")

Note: you may need to restart the kernel to use updated packages.
✅ Libraries and Kagglehub Imported Successfully!


/root/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Directory Structure Setup & Automatic Dataset Download
Vast.ai Instance ထဲမှာ KITTI Dataset ရဲ့ ဖွဲ့စည်းပုံကို သတ်မှတ်ပါမယ်။ Kagglehub ကို အသုံးပြုပြီး `klemenko/kitti-dataset` ကို အလိုအလျောက် ဒေါင်းလုဒ်ဆွဲပြီးနောက် ဆရာ့ရဲ့ မူရင်း Layout ဖြစ်တဲ့ `image_2` (RGB Images)၊ `label_2` (Text Labels) နဲ့ `calib` Folder များထဲသို့ စနစ်တကျ ရွှေ့ပြောင်းပေးသွားမှာ ဖြစ်ပါတယ်။

In [3]:
import os
import shutil
import kagglehub

# ၁။ Folder Structure ကို သတ်မှတ်ခြင်း
DATASET_ROOT = "./data/KITTI/"
IMAGE_DIR = os.path.join(DATASET_ROOT, "training/image_2")
LABEL_DIR = os.path.join(DATASET_ROOT, "training/label_2")
CALIB_DIR = os.path.join(DATASET_ROOT, "training/calib")
OUTPUT_LABEL_DIR = os.path.join(DATASET_ROOT, "training/yolo_labels")

# 💡 [CRITICAL CHECK] ဖိုင်တွေ ရှိပြီးသားလား အရင်ဆုံး စစ်ဆေးခြင်း
# Image, Label, Calib ဖိုဒါတွေထဲမှာ ဖိုင်တွေ အပြည့်အစုံ ရှိနေရင် ဒေါင်းလုဒ်လုပ်ငန်းစဉ်တစ်ခုလုံးကို အစကနေ ကျော်ပါမယ်
if os.path.exists(IMAGE_DIR) and len(os.listdir(IMAGE_DIR)) > 0:
    print("✨ [SMART SKIP] KITTI Dataset ဟာ စက်ထဲမှာ အဆင်သင့် ရှိနေပြီးသား ဖြစ်ပါတယ်ဗျာ!")
    print("📥 ဒေါင်းလုဒ်ထပ်မံဆွဲခြင်းကို အလိုအလျောက် ကျော်လွှားလိုက်ပါပြီ။")
else:
    print("🔍 စက်ထဲတွင် KITTI Dataset ကို မတွေ့ရှိသေးပါ သို့မဟုတ် ဖိုင်များ မပြည့်စုံသေးပါ။ Fresh Setup စတင်ပါမည်...")
    
    # ဖိုင်မရှိသေးမှသာ ဖိုဒါအဟောင်းများကို ရှင်းပြီး အသစ်ပြန်ဆောက်မည်
    if os.path.exists(DATASET_ROOT):
        shutil.rmtree(DATASET_ROOT)

    os.makedirs(IMAGE_DIR, exist_ok=True)
    os.makedirs(LABEL_DIR, exist_ok=True)
    os.makedirs(CALIB_DIR, exist_ok=True)
    os.makedirs(OUTPUT_LABEL_DIR, exist_ok=True)

    # Cache အဟောင်း ဖျက်ခြင်း (ပထမအကြိမ် ဒေါင်းလုဒ်အတွက်သာ)
    cache_root = "/root/.cache/kagglehub"
    if os.path.exists(cache_root):
        print("🧹 Removing broken Kaggle cache to force a real re-download...")
        shutil.rmtree(cache_root)

    print("✅ Folders initialised. Ready for fresh download.")

    # ၃။ Kagglehub သုံးပြီး Dataset ကို အသစ်စက်စက် စတင်ဒေါင်းလုဒ်ဆွဲခြင်း
    print("📥 [REAL DOWNLOAD START] Fetching KITTI Dataset from Kaggle Server... (Please wait 3-5 mins)")
    downloaded_cache_path = kagglehub.dataset_download("klemenko/kitti-dataset")

    print("\n🎉 [DOWNLOAD COMPLETED]!")
    print("Path to downloaded cache files:", downloaded_cache_path)
    print("🔄 Arranging files into your exact original layout...")

    # ၄။ ဒေါင်းလုဒ်ရလာသော ဖိုင်များကို ဆရာ့ရဲ့ မူရင်း Folder များထဲသို့ အရောက်ရွှေ့ပြောင်းခြင်း
    for root, dirs, files in os.walk(downloaded_cache_path):
        if len(files) == 0:
            continue
        root_lower = root.lower()
        
        if "image_2" in root_lower or "images" in root_lower:
            for f in files:
                shutil.move(os.path.join(root, f), os.path.join(IMAGE_DIR, f))
                
        elif "label_2" in root_lower or "labels" in root_lower:
            for f in files:
                shutil.move(os.path.join(root, f), os.path.join(LABEL_DIR, f))
                
        elif "calib" in root_lower or "calibration" in root_lower:
            for f in files:
                shutil.move(os.path.join(root, f), os.path.join(CALIB_DIR, f))

# ၅။ နောက်ဆုံးပိတ် ဖိုင်အရေအတွက် စစ်ဆေးချက်ကို အမြဲတမ်း ပြသပေးမည်
print("\n📦 --- DIRECTORY VERIFICATION STATUS ---")
print(f"📦 Output folder created at: {OUTPUT_LABEL_DIR}")
print(f"🖼️ Verification - Images Found: {len(os.listdir(IMAGE_DIR))} files")
print(f"📝 Verification - Labels Found: {len(os.listdir(LABEL_DIR))} files")
print(f"📐 Verification - Calibration Found: {len(os.listdir(CALIB_DIR))} files")

✨ [SMART SKIP] KITTI Dataset ဟာ စက်ထဲမှာ အဆင်သင့် ရှိနေပြီးသား ဖြစ်ပါတယ်ဗျာ!
📥 ဒေါင်းလုဒ်ထပ်မံဆွဲခြင်းကို အလိုအလျောက် ကျော်လွှားလိုက်ပါပြီ။

📦 --- DIRECTORY VERIFICATION STATUS ---
📦 Output folder created at: ./data/KITTI/training/yolo_labels
🖼️ Verification - Images Found: 7518 files
📝 Verification - Labels Found: 7481 files
📐 Verification - Calibration Found: 7518 files


## 3. Class Mapping Definition
To transform the 8 official object classes from the KITTI dataset into a format readable by the YOLO architecture, we construct a comprehensive dictionary mapping that assigns distinct integer IDs (ranging from 0 to 7) to each category. By avoiding grouping similar types, this approach allows the model to learn fine-grained boundaries between all target objects, including 'Car', 'Van', 'Truck', 'Pedestrian', 'Person_sitting', 'Cyclist', 'Tram', and 'Misc'.


KITTI Dataset တွင် ပါဝင်သော Object Classes စုစုပေါင်း (၈) ခုလုံးကို YOLO Architecture မှ နားလည်နိုင်သော သီးခြား Integer IDs (0 မှ 7 အထိ) အဖြစ် ပြောင်းလဲရန်အတွက် Dictionary Mapping တစ်ခုကို တည်ဆောက်ပါမည်။ ဤနည်းဗျူဟာသည် အုပ်စုတူများကို စုစည်းခြင်းမပြုဘဲ 'Car', 'Van', 'Truck', 'Pedestrian', 'Person_sitting', 'Cyclist', 'Tram' နှင့် 'Misc' စသည့် အမျိုးအစားအားလုံးကို ပိုမိုတိကျစွာ သီးသန့်ခွဲခြားသင်ယူနိုင်ရန်အတွက် ဖြစ်သည်။

In [4]:
# Mapping KITTI string classes to Distinct Integer IDs (Full 8 Classes)
CLASS_MAP = {
    'Car': 0,
    'Van': 1,
    'Truck': 2,
    'Pedestrian': 3,
    'Person_sitting': 4,
    'Cyclist': 5,
    'Tram': 6,
    'Misc': 7
}

print("🚗 Full 8-Class Mapping:", CLASS_MAP)

🚗 Full 8-Class Mapping: {'Car': 0, 'Van': 1, 'Truck': 2, 'Pedestrian': 3, 'Person_sitting': 4, 'Cyclist': 5, 'Tram': 6, 'Misc': 7}


## 4. KITTI to YOLO-3D Converter Engine
This cell serves as the core pipeline engine. It reads each KITTI label file and not only computes the normalized 2D bounding boxes (0-1 format) but also extracts the 3D coordinates ($X, Y, Z$), dimensions ($H, W, L$), and the global orientation (Yaw angle) for all 8 official classes. Finally, it formats and aggregates this data into the strict YOLO-3D target vector format: `[class_id, x, y, w, h, x3d, y3d, z3d, h3d, w3d, l3d, yaw]`.



ဒီ Cell ကတော့ ဤ Pipeline ၏ အသက်သွေးကြော ဖြစ်ပါသည်။ KITTI Label ဖိုင်တစ်ခုချင်းစီကို ဖတ်ပြီး 2D Bounding Box (Normalized 0-1) ကို တွက်ချက်ရုံတင်မကဘဲ၊ တရားဝင် Object Classes ၈ ခုလုံး၏ 3D Coordinates ($X, Y, Z$)၊ Dimensions ($H, W, L$) နှင့် Global Orientation (Yaw angle) တို့ကိုပါ စုစည်းပြီး YOLO-3D ၏ Text Format ဖြစ်သော `[class_id, x, y, w, h, x3d, y3d, z3d, h3d, w3d, l3d, yaw]` အဖြစ်သို့ တိုက်ရိုက်ပြောင်းလဲပေးမည့် Converter Function ဖြစ်သည်။

In [5]:
def convert_kitti_to_yolo3d(label_path, calib_path, img_width, img_height):
    yolo_annotations = []
    
    if not os.path.exists(label_path):
        return yolo_annotations

    with open(label_path, 'r') as f:
        lines = f.readlines()
        
    for line in lines:
        elements = line.strip().split(' ')
        obj_type = elements[0]
        
        # Skip background or unclassified objects
        if obj_type not in CLASS_MAP:
            continue
            
        class_id = CLASS_MAP[obj_type]
        
        # 1. Extract 2D Bounding Box (pixels)
        xmin = float(elements[4])
        ymin = float(elements[5])
        xmax = float(elements[6])
        ymax = float(elements[7])
        
        # Convert to YOLO standard: normalized center_x, center_y, width, height
        x_center = ((xmin + xmax) / 2.0) / img_width
        y_center = ((ymin + ymax) / 2.0) / img_height
        box_w = (xmax - xmin) / img_width
        box_h = (ymax - ymin) / img_height
        
        # 2. Extract 3D Dimensions (Height, Width, Length) in meters
        h3d = float(elements[8])
        w3d = float(elements[9])
        l3d = float(elements[10])
        
        # 3. Extract 3D Camera Coordinates (X, Y, Z - Distance) in meters
        x3d = float(elements[11])
        y3d = float(elements[12])
        z3d = float(elements[13])  # <--- ဒီကောင်က ဆရာလိုချင်တဲ့ Depth/Distance ပါ
        
        # 4. Extract Rotation (Yaw angle)
        yaw = float(elements[14])
        
        # Append into YOLO-3D target vector format
        # [class_id, x, y, w, h, x3d, y3d, z3d, h3d, w3d, l3d, yaw]
        yolo_str = f"{class_id} {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f} {x3d:.4f} {y3d:.4f} {z3d:.4f} {h3d:.4f} {w3d:.4f} {l3d:.4f} {yaw:.4f}"
        yolo_annotations.append(yolo_str)
        
    return yolo_annotations

print("⚙️ Converter Engine Function Compiled Successfully!")

⚙️ Converter Engine Function Compiled Successfully!


## 5. Execution Pipeline (Batch Processing)
At this stage, the pipeline dynamically extracts the resolution of each image across the entire training dataset. It then batch-processes and converts the corresponding KITTI label files into the continuous YOLO-3D format for all 8 official object classes simultaneously. The execution progress is tracked and visualized using a progress bar (`tqdm`).

ယခုအခါတွင်မူ Training Dataset တစ်ခုလုံးတွင် ရှိကြသော ရုပ်ထွက်အရွယ်အစား (Image Resolutions) များကို dynamically ဖတ်ရှုပြီး၊ သက်ဆိုင်ရာ KITTI Label ဖိုင်များအားလုံးကို သတ်မှတ်ထားသော တရားဝင် Object Classes (၈) ခုလုံးအတွက် Batch အလိုက် တစ်ပြိုင်နက်တည်း ပြောင်းလဲပေးသွားမည် ဖြစ်သည်။ အလုပ်လုပ်ဆောင်မှု အခြေအနေကို Progress Bar (`tqdm`) ဖြင့် စောင့်ကြည့်နိုင်မည် ဖြစ်သည်။

In [6]:
# ==============================================================================
# ⚙️ SECTION 5: EXECUTION PIPELINE (BATCH PROCESSING)
# ==============================================================================

%pip install -q opencv-python
import os
import cv2
from tqdm import tqdm

# Get all text files in the KITTI label directory
label_files = [f for f in os.listdir(LABEL_DIR) if f.endswith('.txt')]

# 💡 [SMART SKIP] ပြောင်းပြီးသား output တွေ အပြည့်အစုံ ရှိနေရင် အချိန်မကုန်အောင် အလိုအလျောက် ကျော်သွားပါမည်
if os.path.exists(OUTPUT_LABEL_DIR) and len(os.listdir(OUTPUT_LABEL_DIR)) == len(label_files):
    print(f"✨ [SMART SKIP] ဖိုင်ပေါင်း {len(label_files)} ဖိုင်လုံးကို YOLO-3D ဖော်မတ်သို့ ပြောင်းလဲပြီးသား ဖြစ်သဖြင့် ကျော်လွှားလိုက်ပါပြီဗျာ။")
else:
    print(f"🔄 Processing {len(label_files)} files...")

    for file_name in tqdm(label_files):
        base_name = os.path.splitext(file_name)[0]
        
        img_path = os.path.join(IMAGE_DIR, base_name + ".png")
        lbl_path = os.path.join(LABEL_DIR, file_name)
        calib_path = os.path.join(CALIB_DIR, base_name + ".txt")
        
        # Output လမ်းကြောင်း
        output_file_path = os.path.join(OUTPUT_LABEL_DIR, file_name)
        
        # 💡 တစ်ဖိုင်ချင်းစီအလိုက် ပြောင်းပြီးသားရှိရင် ကျော်ရန် (VRAM နှင့် အချိန် ချွေတာရန်)
        if os.path.exists(output_file_path):
            continue
            
        try:
            # Read image to get dynamic width and height
            img = cv2.imread(img_path)
            if img is None:
                continue
            h, w, _ = img.shape
            
            # Run conversion
            yolo_data = convert_kitti_to_yolo3d(lbl_path, calib_path, w, h)
            
            # Write to output folder
            with open(output_file_path, 'w') as out_f:
                out_f.write('\n'.join(yolo_data))
        except Exception as e:
            # 💡 စာဖတ်နေစဉ် သို့မဟုတ် ပြောင်းလဲစဉ် ပြဿနာရှိပါက ကုဒ်ကြီး ဒေါင်းမသွားအောင် ကျော်သွားခြင်း
            print(f"⚠️ Error processing file {file_name}: {str(e)}")
            continue

    print("✨ All KITTI Labels have been converted to YOLO-3D Matrix format successfully!")

Note: you may need to restart the kernel to use updated packages.
✨ [SMART SKIP] ဖိုင်ပေါင်း 7481 ဖိုင်လုံးကို YOLO-3D ဖော်မတ်သို့ ပြောင်းလဲပြီးသား ဖြစ်သဖြင့် ကျော်လွှားလိုက်ပါပြီဗျာ။


## 6. Dataset Train/Validation Split
Prior to launching the model training on the Vast.ai GPU cloud, it is essential to partition the prepared images and label files into a Training Set (80%) and a Validation Set (20%) using a randomized split. This split is critical for evaluating the model's generalization capability and detecting potential overfitting via rigorous performance metrics.

Vast.ai GPU Cloud ပေါ်တွင် မော်ဒယ်အား စတင်သင်ယူခြင်း (Training) မပြုလုပ်မီ၊ ပြောင်းလဲပြင်ဆင်ထားသော Image နှင့် Label ဖိုင်များအားလုံးကို Train Set (80%) နှင့် Validation Set (20%) ဟူ၍ Random ခွဲခြားပေးရန် လိုအပ်သည်။ ဤသို့ ခွဲခြားခြင်းဖြင့် မော်ဒယ်၏ အထွေထွေခန့်မှန်းနိုင်စွမ်း (Generalization Ability) ကို စစ်ဆေးနိုင်ပြီး Overfitting ဖြစ်ပွားမှုကို တိကျသော Evaluation Metrics များဖြင့် စောင့်ကြည့်ထိန်းချုပ်နိုင်မည် ဖြစ်သည်။

In [7]:
import random
import glob
import os

# Get all base names of images (e.g., '000001')
all_images = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.png")))
all_basenames = [os.path.basename(f).split('.')[0] for f in all_images]

# Shuffle randomly with a fixed seed for reproducibility
random.seed(42)
random.shuffle(all_basenames)

# Calculate split index
split_idx = int(len(all_basenames) * 0.8)
train_list = all_basenames[:split_idx]
val_list = all_basenames[split_idx:]

print(f"📊 Total Dataset: {len(all_basenames)} samples")
print(f"🔹 Training Set List (In-Memory): {len(train_list)} samples ready.")
print(f"🔸 Validation Set List (In-Memory): {len(val_list)} samples ready.")

📊 Total Dataset: 7518 samples
🔹 Training Set List (In-Memory): 6014 samples ready.
🔸 Validation Set List (In-Memory): 1504 samples ready.


## 7. Generating Split Text Files
To enable direct dataset referencing for the YOLO-3D framework and PyTorch DataLoaders, this cell materializes the in-memory lists (`train_list` and `val_list`) generated in Section 6 into actual tracking text files: `train.txt` and `val.txt` within the designated `DATASET_ROOT` directory.


YOLO-3D Framework နှင့် PyTorch DataLoader တို့မှ Dataset အား တိုက်ရိုက် လမ်းညွှန်ဖတ်ရှုနိုင်စေရန်အတွက် အထက်ပါ Code Block 6 မှ ခွဲထုတ်ပေးလိုက်သော Memory ထဲရှိ `train_list` နှင့် `val_list` စာရင်းများကို `train.txt` နှင့် `val.txt` စာသားဖိုင်အစစ်အမှန်များအဖြစ် သတ်မှတ်ထားသော ဖိုဒါလမ်းကြောင်း (`DATASET_ROOT`) ထဲတွင် အပြီးသတ် ရေးသားသိမ်းဆည်းပေးမည် ဖြစ်သည်။

In [8]:
# Create text files containing paths to images
def write_split_file(filename, basename_list):
    file_path = os.path.join(DATASET_ROOT, filename)
    with open(file_path, 'w') as f:
        for name in basename_list:
            # Pointing to the image path relative to root
            f.write(f"./training/image_2/{name}.png\n")
    print(f"💾 Created and Saved: {file_path}")

# Code Block 6 မှ ထွက်လာသော memory lists များကို ဖိုင်အဖြစ် တကယ့် Hard Drive ထဲသိမ်းခြင်း
write_split_file("train.txt", train_list)
write_split_file("val.txt", val_list)

💾 Created and Saved: ./data/KITTI/train.txt
💾 Created and Saved: ./data/KITTI/val.txt


## 8. Automated YOLO-3D Configuration (`yaml`) File Creation
This cell programmatically generates the `kitti_3d.yaml` dataset configuration file, which maps the continuous directories and tracks the flow of our custom multi-task labels (including 2D bounding boxes, 3D camera coordinates, and orientation annotations) into the YOLO network. Reflecting our full-scale research configuration, this file declares all 8 official KITTI object classes (`nc: 8`) along with the 3D regression metadata requirements to drive the training sequence.


ဒီ Cell ကတော့ YOLO Network ထဲသို့ ကျွန်တော်တို့၏ Custom Multi-task Labels များ (2D Bounding Boxes + 3D Camera Coordinates + Orientation) သွားမည့် ဒေတာလမ်းကြောင်းများကို တရားဝင် ကြေညာပေးမည့် `kitti_3d.yaml` ဖိုင်ကို Programmatically ဆောက်ပေးမှာ ဖြစ်သည်။ ယခုအခါတွင် ကျွန်တော်တို့၏ သုတေသန ဗျူဟာအသစ်အရ KITTI ၏ တရားဝင် Object Classes (၈) ခုလုံး (`nc: 8`) နှင့် ၎င်းတို့၏ သတ်မှတ်ချက် Meta-data များကို Network ဆီသို့ စနစ်တကျ ချိတ်ဆက်ပေးသွားမည် ဖြစ်သည်။

In [9]:
# ==============================================================================
# ⚙️ SECTION 8: AUTOMATED YOLO-3D CONFIGURATION (YAML) FILE CREATION
# ==============================================================================
import os
import yaml

# Construct the configuration dictionary for FULL 8 CLASSES
yolo_config = {
    'path': os.path.abspath(DATASET_ROOT), # Dataset root folder absolute path
    'train': 'train.txt',                  # Path relative to 'path'
    'val': 'val.txt',                      # Path relative to 'path'
    
    # 💡 [FIXED] 2D & 3D Multi-Task Classes (Full 8 Official KITTI Categories)
    'names': {
        0: 'Car',
        1: 'Van',
        2: 'Truck',
        3: 'Pedestrian',
        4: 'Person_sitting',
        5: 'Cyclist',
        6: 'Tram',
        7: 'Misc'
    },
    
    # Custom Metadata for 3D Geometry Regression Head
    'nc': 8,                               # 💡 [FIXED] Number of classes changed from 3 to 8
    'has_3d_target': True,                 # Flag for custom dataloader to fetch X, Y, Z
    'num_3d_attributes': 7                 # X, Y, Z (Distance), H, W, L, Yaw Angle
}

# Write to custom yaml file
config_yaml_path = os.path.join(DATASET_ROOT, "kitti_3d.yaml")
with open(config_yaml_path, 'w') as yaml_file:
    yaml.dump(yolo_config, yaml_file, default_flow_style=False)

print(f"🚀 YOLO-3D Dataset Config file is successfully saved at: {config_yaml_path}")

🚀 YOLO-3D Dataset Config file is successfully saved at: ./data/KITTI/kitti_3d.yaml


## 9. Verification of Configuration
This cell programmatically inspects and reads back the newly generated `kitti_3d.yaml` configuration file to verify its structural integrity. It ensures that all 8 target object classes, regression attributes, and absolute path pointers are mapped precisely. Execution of this cell marks the absolute completion of the Data Pipeline & Configuration Phase, clearing the track for the upcoming neural network training.

စနစ်တကျ ဖန်တီးပြီးသွားသော `kitti_3d.yaml` Configuration ဖိုင်ကို ပရိုဂရမ်အရ ပြန်လည်ဖတ်ရှုပြီး အတွင်းရှိ Target Classes (၈) ခုလုံး၏ Metadata များနှင့် ဒေတာလမ်းကြောင်းများ (Data Paths) အားလုံး မှန်ကန်မှု ရှိမရှိကို အပြီးသတ် စစ်ဆေးအတည်ပြု (Validation) လုပ်ဆောင်မည် ဖြစ်သည်။ ဤအဆင့် ပြီးဆုံးပါက ဒေတာပြင်ဆင်မှု လုပ်ငန်းစဉ် (Data Pipeline Phase) တစ်ခုလုံး ရာနှုန်းပြည့် အောင်မြင်စွာ ပြီးဆုံးပြီ ဖြစ်သည်။

In [10]:
# ==============================================================================
# 🔍 SECTION 9: VERIFICATION OF CONFIGURATION
# ==============================================================================

# Read and print out the config file contents to ensure all is good
with open(config_yaml_path, 'r') as verify_file:
    print(verify_file.read())

print("🎯 Data Pipeline & Configuration Phase is fully completed!")

has_3d_target: true
names:
  0: Car
  1: Van
  2: Truck
  3: Pedestrian
  4: Person_sitting
  5: Cyclist
  6: Tram
  7: Misc
nc: 8
num_3d_attributes: 7
path: /workspace/kitti-full-yolo3d-thesis/data/KITTI
train: train.txt
val: val.txt

🎯 Data Pipeline & Configuration Phase is fully completed!


## 10. Baseline Model Architecture Design: Custom 3D-YOLO Head
Standard YOLOv8/v7 တွေက 2D bounding boxes (`x, y, w, h`) ကိုပဲ Output ထုတ်ပေးတာ ဖြစ်လို့၊ ကျွန်တော်တို့ ရွေးချယ်ထားတဲ့ Baseline Paper အတိုင်း 3D Attributes (Dimensions: H, W, L၊ Location: X, Y, Z၊ Orientation: Yaw angle) စုစုပေါင်း 7 တန်ဖိုးကို တွက်ချက်ပေးမယ့် Parallel Regression Head တစ်ခုကို PyTorch သုံးပြီး တည်ဆောက်ပါမယ်။

In [11]:
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
import torch
import torch.nn as nn

# 1. Image (3ch) မှ Features (256ch) ပြောင်းပေးမည့် Mini-Backbone
class YOLOBackboneBridge(nn.Module):
    def __init__(self):
        super(YOLOBackboneBridge, self).__init__()
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),   # 640x640 -> 320x320
            nn.BatchNorm2d(32),
            nn.SiLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # 320x320 -> 160x160
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), # 160x160 -> 80x80
            nn.BatchNorm2d(128),
            nn.SiLU(),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),# 80x80 -> 40x40 (Matches target)
            nn.BatchNorm2d(256),
            nn.SiLU()
        )
    def forward(self, x):
        return self.feature_extractor(x)

# 2. Complete End-to-End Baseline Model
class FullYOLO3DModel(nn.Module):
    def __init__(self, num_classes=3):
        super(FullYOLO3DModel, self).__init__()
        self.backbone = YOLOBackboneBridge()
        
        # Heads
        self.cls_head = nn.Conv2d(256, num_classes, kernel_size=3, padding=1)
        self.box2d_head = nn.Conv2d(256, 4, kernel_size=3, padding=1)
        self.box3d_head = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 7, kernel_size=1)
        )
        
    def forward(self, x):
        features = self.backbone(x) # 3 channels -> 256 channels
        return {
            'cls': self.cls_head(features),
            'box2d': self.box2d_head(features),
            'box3d': self.box3d_head(features)
        }

print("✅ Full Baseline Model with Backbone Bridge Compiled!")

Note: you may need to restart the kernel to use updated packages.
✅ Full Baseline Model with Backbone Bridge Compiled!


## 11. Multi-Task Loss Function (Depth-Aware Distance Loss)
မဟာဘွဲ့အတွက် ဆရာ့ရဲ့ အဓိက သုတေသနပစ်မှတ်ဖြစ်တဲ့ "Distance/Depth (Z)" ကို Baseline Paper အတိုင်း တိတိကျကျ သင်ယူနိုင်ဖို့အတွက် multi-task loss function တစ်ခုကို သတ်မှတ်ပါမယ်။ 2D တွက်ချက်မှုအတွက် CIoU / SmoothL1 ကို သုံးပြီး 3D ရဲ့ အကွာအဝေးအတွက် Depth-weighted Smooth L1 Loss ကို သုံးပါမယ်။

In [12]:
class YOLO3DLoss(nn.Module):
    def __init__(self):
        super(YOLO3DLoss, self).__init__()
        self.cls_loss = nn.CrossEntropyLoss()
        self.l1_loss = nn.SmoothL1Loss(reduction='mean')
        
    def forward(self, predictions, targets):
        """
        predictions: Model က ခန့်မှန်းချက် (Dict format) -> cls shape: [16, 3, 40, 40]
        targets: Target Ground Truth Matrix          -> cls shape: [16, 1, 40, 40]
        """
        # --- FIXED LINE: .squeeze(1) သုံးပြီး [16, 1, 40, 40] မှ [16, 40, 40] သို့ ပြောင်းလဲခြင်း ---
        target_cls = targets['cls'].squeeze(1) 
        
        # 1. 2D & Classification Loss
        loss_cls = self.cls_loss(predictions['cls'], target_cls)
        loss_box2d = self.l1_loss(predictions['box2d'], targets['box2d'])
        
        # 2. 3D Distance Regression Loss
        pred_3d = predictions['box3d']
        target_3d = targets['box3d']
        loss_box3d = self.l1_loss(pred_3d, target_3d)
        
        # Total Balanced Loss
        total_loss = loss_cls + loss_box2d + (2.0 * loss_box3d)
        
        return total_loss, loss_box2d, loss_box3d

print("⚖️ Fixed Loss Function Initialized with Target Squeeze Logic.")

⚖️ Fixed Loss Function Initialized with Target Squeeze Logic.


## 12. Vast.ai RTX 4090 Engine Activation (Training Loop Stub)
ဆရာ့ရဲ့ RTX 4090 GPU ထဲကို Model ရော Data ပါ တင်ပြီး `torch.cuda.amp` (Automatic Mixed Precision - FP16) ကို သုံးကာ အမြန်ဆုံးနှုန်းနဲ့ စတင် Training မောင်းနှင်မယ့် အပိုင်းဖြစ်ပါတယ်။ 24GB VRAM ရှိတဲ့အတွက် `batch_size=64` ကို စိတ်ချလက်ချ မောင်းနှင်နိုင်ပါတယ်။

In [13]:
# Check GPU Availability (Vast.ai RTX 4090 confirmation)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Utilizing Device: {device}")

if torch.cuda.is_available():
    print(f"🔥 GPU Model Name: {torch.cuda.get_device_name(0)}")
    print(f"💾 Total VRAM Allocated: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

# Global Hyperparameters
epochs = 10
learning_rate = 0.001
criterion = YOLO3DLoss()
scaler = torch.cuda.amp.GradScaler() # For FP16 Mixed Precision on RTX 4090

print("\n🏋️ Training Hardware & Scalar Engine Configuration Ready!")

🚀 Utilizing Device: cuda
🔥 GPU Model Name: NVIDIA GeForce RTX 4090
💾 Total VRAM Allocated: 23.52 GB

🏋️ Training Hardware & Scalar Engine Configuration Ready!


/tmp/ipykernel_2379/3926728326.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # For FP16 Mixed Precision on RTX 4090


## 13. Proposed Innovation: Coordinate-Spatial Attention Module (CSAM)
ဒီ Module ကတော့ မဟာဘွဲ့အတွက် ကျွန်တော်တို့ အဆိုပြုတဲ့ Novelty အပိုင်းဖြစ်ပါတယ်။ 2D Features တွေကို Horizontal ($X$) နဲ့ Vertical ($Y$) Direction အလိုက် Global Pooling လုပ်ပြီး ခွဲခြမ်းစိတ်ဖြာတဲ့အတွက်၊ ကင်မရာရဲ့ အနေအထားအရ ပုံရဲ့ အောက်ခြေနားက Object တွေက ပိုနီးပြီး အပေါ်ဘက်က Object တွေက ပိုဝေးတယ်ဆိုတဲ့ Spatial Geometry Dependency ကို Model က ပိုမိုသိရှိလာစေမှာ ဖြစ်ပါတယ်။

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CoordinateSpatialAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super(CoordinateSpatialAttention, self).__init__()
        # Reduction ratio သုံးပြီး Computation cost ကို လျှော့ချပါမယ်
        self.fc1 = nn.Conv2d(channels, channels // reduction, kernel_size=1)
        self.bn1 = nn.BatchNorm2d(channels // reduction)
        self.act = nn.SiLU() # YOLO series မှာ အသုံးများတဲ့ Activation Function ဖြစ်ပါတယ်
        
        self.fc_x = nn.Conv2d(channels // reduction, channels, kernel_size=1)
        self.fc_y = nn.Conv2d(channels // reduction, channels, kernel_size=1)
        
    def forward(self, x):
        b, c, h, w = x.size()
        
        # 1. X နှင့် Y Direction အလိုက် ဉာဏ်ရည်ပြည့်မီစွာ Feature Map ကို ညှစ်ထုတ်ခြင်း (Pooling)
        pool_x = F.adaptive_avg_pool2d(x, (h, 1)) # Shape: (B, C, H, 1)
        pool_y = F.adaptive_avg_pool2d(x, (1, w)).transpose(2, 3) # Shape: (B, C, W, 1)
        
        # 2. Features များကို ပေါင်းစပ်ပြီး Spatial Spatial Interaction ရှာဖွေခြင်း
        concat = torch.cat([pool_x, pool_y], dim=2) # Shape: (B, C, H+W, 1)
        mip = self.act(self.bn1(self.fc1(concat)))
        
        # Split ပြန်လုပ်ခြင်း
        x_att, y_att = torch.split(mip, [h, w], dim=2)
        y_att = y_att.transpose(2, 3)
        
        # 3. Sigmoid သုံးပြီး Attention Weights (0 to 1) ပြောင်းလဲခြင်း
        a_x = torch.sigmoid(self.fc_x(x_att))
        a_y = torch.sigmoid(self.fc_y(y_att))
        
        # Input features ကို Attention weights များနှင့် နှောက်ယှက်မြှောက်ပေးခြင်း (Refinement)
        out = x * a_x * a_y
        return out

print("💎 Proposed Coordinate Spatial Attention Module (CSAM) Compiled Successfully!")

💎 Proposed Coordinate Spatial Attention Module (CSAM) Compiled Successfully!


## 14. Integrating Innovation Into the Enhanced YOLO-3D Architecture
ယခုအဆင့်မှာတော့ ကျွန်တော်တို့ အသစ်ထွင်လိုက်တဲ့ `CoordinateSpatialAttention` module ကို ရှေ့က ဆောက်ခဲ့တဲ့ `YOLO3DHead` ထဲမှာ ဝင်ရောက် Plug-in လုပ်ပြီး "Enhanced YOLO-3D Head" အဖြစ် ပြောင်းလဲဖွဲ့စည်းပါမယ်။

In [15]:
import torch
import torch.nn as nn

# 1. Image (3ch) မှ Features (256ch) ပြောင်းပေးမည့် Mini-Backbone
class YOLOBackboneBridge(nn.Module):
    def __init__(self):
        super(YOLOBackboneBridge, self).__init__()
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),   # 640x640 -> 320x320
            nn.BatchNorm2d(32),
            nn.SiLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # 320x320 -> 160x160
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), # 160x160 -> 80x80
            nn.BatchNorm2d(128),
            nn.SiLU(),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),# 80x80 -> 40x40 (Matches target)
            nn.BatchNorm2d(256),
            nn.SiLU()
        )
    def forward(self, x):
        return self.feature_extractor(x)

# 2. Complete End-to-End Baseline Model
class FullYOLO3DModel(nn.Module):
    def __init__(self, num_classes=3):
        super(FullYOLO3DModel, self).__init__()
        self.backbone = YOLOBackboneBridge()
        
        # Heads
        self.cls_head = nn.Conv2d(256, num_classes, kernel_size=3, padding=1)
        self.box2d_head = nn.Conv2d(256, 4, kernel_size=3, padding=1)
        self.box3d_head = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 7, kernel_size=1)
        )
        
    def forward(self, x):
        features = self.backbone(x) # 3 channels -> 256 channels
        return {
            'cls': self.cls_head(features),
            'box2d': self.box2d_head(features),
            'box3d': self.box3d_head(features)
        }

print("✅ Full Baseline Model with Backbone Bridge Compiled!")
# 3. Complete End-to-End Proposed Model (With Attention Module)
class FullEnhancedYOLO3DModel(nn.Module):
    def __init__(self, num_classes=3):
        super(FullEnhancedYOLO3DModel, self).__init__()
        self.backbone = YOLOBackboneBridge()
        # ဆရာ့ရဲ့ Custom Attention Module ကို Backbone အထွက်မှာ တပ်ဆင်ပါတယ်
        self.attention_block = CoordinateSpatialAttention(channels=256)
        
        # Heads
        self.cls_head = nn.Conv2d(256, num_classes, kernel_size=3, padding=1)
        self.box2d_head = nn.Conv2d(256, 4, kernel_size=3, padding=1)
        self.box3d_head = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 7, kernel_size=1)
        )
        
    def forward(self, x):
        features = self.backbone(x)
        refined_features = self.attention_block(features) # Apply Attention
        return {
            'cls': self.cls_head(refined_features),
            'box2d': self.box2d_head(refined_features),
            'box3d': self.box3d_head(refined_features)
        }

print("💎 Full Enhanced Model with Attention & Backbone Compiled!")

✅ Full Baseline Model with Backbone Bridge Compiled!
💎 Full Enhanced Model with Attention & Backbone Compiled!


## 15. Implementing Keras-Style Callbacks in PyTorch Wrapper
Baseline Model ကော Proposed (Enhanced) Model ပါ ဗားရှင်းနှစ်ခုလုံးကို စနစ်တကျ Training ပေးပြီး Validation Multi-task Losses တွေကို Epoch အလိုက် မှတ်တမ်းတင် (Log) မည့် Core Training Function ကို ရေးသားပါမယ်။ RTX 4090 ရဲ့ စွမ်းဆောင်ရည်ကို သုံးဖို့ `torch.cuda.amp` (Mixed Precision) ကို အပြည့်အဝ ထည့်သွင်းထားပါတယ်။

In [ ]:
import time
import torch

class DesignPatternModel:
    def __init__(self, model):
        self.model = model.to(device)
        self.criterion = None
        self.optimizer = None
        self.scaler = None
        self.scheduler = None # Learning Rate Scheduler ဖမ်းရန်
        self.history = {'train_loss': [], 'val_loss': [], 'val_depth_mae': []}
        
        # Early Stopping Internal Variables
        self.patience = 3
        self.patience_counter = 0
        self.best_val_loss = float('inf')

    def compile(self, optimizer, criterion, scaler, scheduler=None, patience=3):
        """ Keras-Style Compile: Hyperparameters များကို ဤနေရာတွင် စုစည်းသတ်မှတ်ခြင်း """
        self.optimizer = optimizer
        self.criterion = criterion
        self.scaler = scaler
        self.scheduler = scheduler # e.g., ReduceLROnPlateau
        self.patience = patience   # Early Stopping အတွက် စောင့်ဆိုင်းမည့် Epoch အရေအတွက်
        print(f"⚙️ Model Compiled with EarlyStopping(patience={patience}) and LR Scheduler!")

    def fit(self, train_loader, val_loader, epochs=10):
        """ ဆရာအကြိုက်ဆုံး ကွင်းဆက်ဖြစ်သည့် ရှင်းလင်းသော fit() interface """
        print(f"🏋️ Starting Training on: {torch.cuda.get_device_name(0)}")
        
        for epoch in range(epochs):
            start_time = time.time()
            
            # 1. Train & Evaluate
            train_loss = self._train_one_epoch(train_loader)
            val_loss, val_depth_mae = self.evaluate(val_loader)
            
            # Log Metrics
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['val_depth_mae'].append(val_depth_mae)
            
            # Current Learning Rate ရှာဖွေခြင်း
            current_lr = self.optimizer.param_groups[0]['lr']
            elapsed = time.time() - start_time
            
            print(f"Epoch [{epoch+1}/{epochs}] ({elapsed:.1f}s) -> "
                  f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
                  f"Depth MAE: {val_depth_mae:.2f}m | LR: {current_lr:.6f}")
            
            # -----------------------------------------------------------------
            # 2. AUTOMATIC CALLBACKS ENGINE (နောက်ကွယ်မှ အလိုအလျောက် ထိန်းကျောင်းမှု)
            # -----------------------------------------------------------------
            
            # (A) Learning Rate Scheduler Step (Val Loss ကို ကြည့်ပြီး LR ချခြင်း)
            if self.scheduler is not None:
                # PyTorch Native Scheduler အား Target Pass ပေးခြင်း
                if isinstance(self.scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                    self.scheduler.step(val_loss)
                else:
                    self.scheduler.step()
            
            # (B) Early Stopping Logic
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.patience_counter = 0 # Val Loss ကျနေသရွေ့ counter ကို resets ချမည်
                # Best Model State ကို Save ထားခြင်း (Optional)
                torch.save(self.model.state_dict(), 'best_thesis_model.pth')
            else:
                self.patience_counter += 1
                print(f"⚠️ Val Loss didn't improve. EarlyStopping Counter: {self.patience_counter}/{self.patience}")
                
            if self.patience_counter >= self.patience:
                print(f"🛑 [Early Stopping] Training stopped automatically at Epoch {epoch+1} to prevent Overfitting!")
                break
                
        return self.history

    def evaluate(self, val_loader):
        self.model.eval()
        running_val_loss = 0.0
        total_depth_error = 0.0
        total_samples = 0
        
        with torch.no_grad():
            for images, targets in val_loader:
                images = images.to(device)
                targets_gpu = {k: v.to(device) for k, v in targets.items()}
                
                outputs = self.model(images)
                loss, _, _ = self.criterion(outputs, targets_gpu)
                running_val_loss += loss.item()
                
                pred_z = outputs['box3d'][:, 2, :, :]
                target_z = targets_gpu['box3d'][:, 2, :, :]
                total_depth_error += torch.abs(pred_z - target_z).mean().item()
                total_samples += 1
                
        return running_val_loss / len(val_loader), total_depth_error / total_samples

    def _train_one_epoch(self, train_loader):
        self.model.train()
        running_train_loss = 0.0
        for images, targets in train_loader:
            images = images.to(device)
            targets_gpu = {k: v.to(device) for k, v in targets.items()}
            
            self.optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                outputs = self.model(images)
                loss, _, _ = self.criterion(outputs, targets_gpu)
                
            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()
            running_train_loss += loss.item()
            
        return running_train_loss / len(train_loader)

print("🎯 Advanced Keras-Style Wrapper with EarlyStopping and LR Scheduler Ready!")

## 16-A. Production-Grade KITTI 3D PyTorch Dataset & Dataloader
Phase 2 က ဆောက်ခဲ့တဲ့ train.txt၊ val.txt ဖိုင်တွေနဲ့ yolo_labels ဖိုင်တွေကို အသုံးပြုပြီး Real PNG Images တွေနဲ့ 3D Geometry Targets တွေကို Vast.ai ရဲ့ GPU VRAM ပေါ် တိုက်ရိုက် Tensor Mapping လုပ်ပေးမယ့် Real Dataset Class ကို တည်ဆောက်ပါမယ်။

In [17]:
import torch
from torch.utils.data import Dataset, DataLoader

class KITTI3DDataset(Dataset):
    def __init__(self, root_dir, split_file, img_size=640):
        self.root_dir = root_dir
        self.img_size = img_size
        
        # Read train.txt or val.txt to get specific image path list
        split_path = os.path.join(root_dir, split_file)
        with open(split_path, 'r') as f:
            self.image_files = [line.strip() for line in f.readlines() if line.strip()]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        # 1. Load Real Image from Disk
        relative_img_path = self.image_files[idx]
        img_path = os.path.join(self.root_dir, relative_img_path)
        
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        
        # Resize to YOLO target size and normalize
        img_resized = cv2.resize(img, (self.img_size, self.img_size))
        img_tensor = torch.from_numpy(img_resized).permute(2, 0, 1).float() / 255.0

        # 2. Locate and Load Corresponding YOLO-3D Label Text File
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(self.root_dir, "training/yolo_labels", f"{base_name}.txt")
        
        # Fixed spatial grid arrays for custom 3D regression head (e.g., 40x40 spatial grid representation)
        cls_target = torch.zeros((1, 40, 40), dtype=torch.long)
        box2d_target = torch.zeros((4, 40, 40), dtype=torch.float32)
        box3d_target = torch.zeros((7, 40, 40), dtype=torch.float32)

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()
            
            for line in lines:
                data = list(map(float, line.strip().split()))
                if len(data) < 12: continue
                
                class_id = int(data[0])
                # Grid indexing mapping (simplified anchor mapping for cell presentation)
                gx = int(data[1] * 39)
                gy = int(data[2] * 39)
                
                # Assign to multi-task tensor spatial matrix
                cls_target[0, gy, gx] = class_id
                box2d_target[:, gy, gx] = torch.tensor(data[1:5])
                box3d_target[:, gy, gx] = torch.tensor(data[5:12]) # X, Y, Z, H, W, L, Yaw

        targets = {
            'cls': cls_target,
            'box2d': box2d_target,
            'box3d': box3d_target
        }
        
        return img_tensor, targets

print("📦 True KITTI 3D Production Dataloader Compiled and Configured!")

📦 True KITTI 3D Production Dataloader Compiled and Configured!


## 16-B. Executing Ablation Studies (Running on Real KITTI Dataset via RTX 4090)
ယခုအခါမှာတော့ Synthetic နေရာမှာ အထက်က တည်ဆောက်ခဲ့တဲ့ တကယ့် KITTI Train & Val Dataloaders တွေကို ထည့်သွင်းပြီး RTX 4090 ရဲ့ GPU Resource ကို အပြည့်အဝ အသုံးပြုကာ Baseline Model ကော ကျွန်တော်တို့ရဲ့ Custom Enhanced Model ပါ နှစ်ခုလုံးကို Real-world Benchmark Training စတင် စမ်းသပ်မောင်းနှင်ပါမယ်။

In [ ]:
print("\n⚡ Step 1: Hooking up real KITTI Dataset pipelines...")
# Initialize Datasets
kitti_train_dataset = KITTI3DDataset(root_dir=DATASET_ROOT, split_file="train.txt", img_size=640)
kitti_val_dataset = KITTI3DDataset(root_dir=DATASET_ROOT, split_file="val.txt", img_size=640)

# OOM Optimization: Batch Size ကို 16 သို့ လျှော့ချပြီး VRAM room ချန်ထားခြင်း
OPTIMIZED_BATCH_SIZE = 16
train_loader = DataLoader(kitti_train_dataset, batch_size=OPTIMIZED_BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(kitti_val_dataset, batch_size=OPTIMIZED_BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# ==========================================
# 1. BASELINE MODEL RUN
# ==========================================
print("\n=== 1. Training Baseline Model ===")

# Model Creation
base_raw_model = FullYOLO3DModel(num_classes=3)
baseline_model = DesignPatternModel(base_raw_model)
adamw_optimizer = torch.optim.AdamW(base_raw_model.parameters(), lr=0.001, weight_decay=1e-4)

# PyTorch Native LR Scheduler: Val Loss မကျရင် Learning Rate ကို 10 ဆ လျှော့ချမည့်စနစ်
plateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    adamw_optimizer, mode='min', factor=0.1, patience=2, verbose=True
)

# Model Compile
baseline_model.compile(
    optimizer=adamw_optimizer,
    criterion=YOLO3DLoss(),
    scaler=torch.cuda.amp.GradScaler(),
    scheduler=plateau_scheduler,
    patience=3
)

# 5. Keras-Style Fit Call (လုံးဝ သပ်ရပ်သွားပါပြီ ဆရာ)
baseline_history = baseline_model.fit(train_loader, val_loader, epochs=15)

# ==========================================
# 2. PROPOSED ENHANCED MODEL RUN
# ==========================================
print("\n=== 2. Training Proposed Enhanced Model ===")

# Model Creation
proposed_raw_model = FullEnhancedYOLO3DModel(num_classes=3)
proposed_model = DesignPatternModel(proposed_raw_model)
adamw_optimizer = torch.optim.AdamW(proposed_model.parameters(), lr=0.001, weight_decay=1e-4)

# PyTorch Native LR Scheduler: Val Loss မကျရင် Learning Rate ကို 10 ဆ လျှော့ချမည့်စနစ်
plateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    adamw_optimizer, mode='min', factor=0.1, patience=2, verbose=True
)

# Model Compile
proposed_model.compile(
    optimizer=adamw_optimizer,
    criterion=YOLO3DLoss(),
    scaler=torch.cuda.amp.GradScaler(),
    scheduler=plateau_scheduler,
    patience=3
)

# Model Fit
proposed_history = proposed_model.fit(train_loader, val_loader, epochs=15)

## 17. Generating the Final Thesis Ablation Matrix (Result Visualization)
သုတေသနနှစ်ခုလုံး ပြီးသွားတဲ့အခါ ကျမ်းစာအုပ် (Thesis Book) ထဲမှာ ထည့်သွင်းရမည့် အကွာအဝေးတွက်ချက်မှု တိကျမှု နှိုင်းယှဉ်ချက် Table Frame ကို Plot ထုတ်ပြီး Validation Result အဖြစ် သိမ်းဆည်းပါမယ်။

In [21]:
# Extract the best (lowest) Depth Mean Absolute Error from both histories
best_baseline_mae = min(baseline_history['val_depth_mae'])
best_proposed_mae = min(proposed_history['val_depth_mae'])

# Calculate the Accuracy Improvement Percentage
improvement = ((best_baseline_mae - best_proposed_mae) / best_baseline_mae) * 100

ablation_matrix = {
    "Model Architecture": ["Baseline YOLO-3D (YOLOv7-3D Concept)", "Proposed Enhanced YOLO-3D (Geometry-Guided)"],
    "Spatial Attention Module": ["❌ None", "✅ Coordinate-Spatial Attention (CSAM)"],
    "Best Distance Error (MAE)": [f"{best_baseline_mae:.3f} meters", f"{best_proposed_mae:.3f} meters"],
    "Accuracy Boost": ["-", f"+ {improvement:.2f}% Improvement"]
}

df_result = pd.DataFrame(ablation_matrix)
# Save to CSV for your Thesis Document
df_result.to_csv("Thesis_Ablation_Matrix_Results.csv", index=False)

print("📝 --- FINAL THESIS ABLATION MATRIX --- 📝")
display(df_result)

📝 --- FINAL THESIS ABLATION MATRIX --- 📝


,Model Architecture,Spatial Attention Module,Best Distance Error (MAE),Accuracy Boost
0,Baseline YOLO-3D (YOLOv7-3D Concept),❌ None,0.080 meters,-
1,Proposed Enhanced YOLO-3D (Geometry-Guided),✅ Coordinate-Spatial Attention (CSAM),0.075 meters,+ 5.99% Improvement
